# Canonical Model 05: Modern PEST Setup

This notebook shows the **declarative calibration facade**: `model.pest(...)` returns a calibration you drive with a few readable lines — `parameterize`, `observe`, `forecast`, `build` — that compile straight to native `pyemu.utils.PstFrom`. No hand-rolled template files, and pyEMU's own `apply_list_and_array_pars` drives the forward run.

We calibrate the **canonical valley model itself**. Its hydraulic conductivity is deliberately reset to a wrong starting value (3x too transmissive), and the head observations are sampled from the true K field, so there is real work for calibration to do.

In [ ]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
import myflopy as mf
from canonical_notebook_style import notebook_header
from myflopy.modflow.mf6.canonical_calibration import build_canonical_calibration_demo

notebook_header(
    '05',
    'Modern PEST Setup',
    'Declare parameters, observations, and forecasts, then build a native PEST++ control file.',
)

## 1. Build the canonical calibration demo

`build_canonical_calibration_demo` builds the canonical valley model as the synthetic **truth**, samples head observations from many wells across the valley floor (plus one down-valley head forecast near the lake), then resets K to a wrong, uniformly-too-high starting value. The returned `model` is the **starting** model; the targets carry the truth-derived measured heads.

In [ ]:
import shutil
artifact_root = Path('../artifacts/canonical_pest_modern')
shutil.rmtree(artifact_root, ignore_errors=True)  # clean rebuild: a stale pest/ template inside the model dir makes PstFrom recurse
artifact_root.mkdir(parents=True, exist_ok=True)

demo = build_canonical_calibration_demo(artifact_root / 'setup_model')

pd.Series({
    'cells': int(demo.model.vor.ncpl),
    'head_observations': len(demo.head_targets.to_long()),
    'observation_wells': demo.head_targets.locations_gdf.shape[0],
    'start_K_factor': demo.start_k_factor,
    'forecast_points': demo.forecast_targets.locations_gdf.shape[0],
}, name='calibration demo')

## 2. Declare parameters

`parameterize(target, ...)` records what to adjust. `bounds` are the multiplier range; `physical` clamps the *final* model value after multiplying (so calibration can't push K somewhere nonphysical). Here we adjust a single constant K multiplier and a constant recharge multiplier. The canonical model is multi-layer, so one constant K multiplier scales every layer's K file together.

> Spatial pilot points for array K need a Voronoi cell spatial reference and are coming in a later phase; `constant` and `zone` work today.

In [ ]:
# Defaults to <model workspace>/pest/canonical_demo (pass workspace= to override).
cal = demo.model.pest('canonical_demo', start_datetime='2024-01-01')

cal.parameterize('k',        style='constant', bounds=(0.05, 2.0), physical=(0.01, 300.0))
cal.parameterize('recharge', style='constant', bounds=(0.3, 3.0),  physical=(0.0, 1e-2))

## 3. Observations and forecasts

`observe` registers history-matching targets; `forecast` registers predictions of interest. Both take the **same** myflopy target objects — a forecast is simply an observation you predict but never match, so it is set to zero weight and recorded as a PEST++ forecast for later uncertainty analysis.

In [ ]:
cal.observe(demo.head_targets)
cal.forecast(demo.forecast_targets)

## 4. Review the resolved configuration (before building)

`cal.settings()` is the anti-black-box: it prints exactly what was declared so you can sanity-check before committing to a build.

In [ ]:
print(cal.settings())

## 5. Build the control file

`build` externalizes the model inputs, registers parameters natively, wires the forward run, and writes the `.pst`. With `noptmax=0` the run is configured to evaluate the model once and compute residuals — the standard sanity check. `settings()` now also reports the control-file counts.

In [ ]:
pst = cal.build('canonical_demo.pst', noptmax=0)
print(cal.settings())

## 6. Inspect the generated PEST control problem

Everything below is read from the actual `.pst` pyEMU wrote — the parameter groups, bounds, and observation weights PEST++ will use.

In [ ]:
display(cal.settings().parameter_frame())
display(
    pst.parameter_data[['parnme', 'pargp', 'partrans', 'parval1', 'parlbnd', 'parubnd']]
    .groupby('pargp').agg(
        parameters=('parnme', 'count'),
        transform=('partrans', 'first'),
        lower_bound=('parlbnd', 'min'),
        upper_bound=('parubnd', 'max'),
    )
)

## 7. Validate the forward run

Every PEST iteration calls `forward_run.py`. Running it once directly is the most important setup check: it must apply the parameter multipliers, run MODFLOW, and regenerate the simulated-observation files. Note that `apply_list_and_array_pars` — pyEMU's native apply — is present, not stripped.

In [ ]:
import subprocess, sys

template = cal.template_workspace
result = subprocess.run([sys.executable, 'forward_run.py'], cwd=template,
                        capture_output=True, text=True)
assert result.returncode == 0, result.stdout + '\n' + result.stderr

forward_run_text = (template / 'forward_run.py').read_text()
pd.Series({
    'forward_run_returncode': result.returncode,
    'apply_list_and_array_pars_present': 'apply_list_and_array_pars' in forward_run_text,
    'mult2model_info_written': (template / 'mult2model_info.csv').exists(),
    'simulated_heads_regenerated': (template / 'hds_simulated_heads.csv').exists(),
}, name='forward-run validation')

## What's next

- The control file, `forward_run.py`, templates, and multiplier files now form a self-contained PEST++ setup.
- Drop to the raw pyEMU objects any time with `cal.pf` and `cal.pst`.
- **Notebook 06** runs PESTPP-IES on this exact setup with one line (`cal.run_ies(...)`) and assesses the result — phi convergence, ensembles vs observations, and posterior forecast uncertainty.